# M2-B1 — Audit Banque Eckmühl + industrialisation pipe

> **Mission** : auditer le dataset German Credit fourni par Eckmühl, identifier
> les risques qualité et éthiques, construire un Pipeline scikit-learn
> réutilisable, persister en Parquet, documenter dans une datasheet.

Auteur·rice : `Tom Carpentier` — Date : `16/06/2026`

**Conventions** :
- `random_state=42` partout
- Pas de `print` (utiliser `display()` ou laisser la cellule retourner)
- `pathlib.Path` sur les chemins

## 0. Setup

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

RANDOM_STATE = 42
DATA_DIR = Path("../data")
RAW_PATH = DATA_DIR / "german_credit_raw.csv"
CLEAN_PATH = DATA_DIR / "german_credit_clean.parquet"

sns.set_theme(style="whitegrid")

## 1. Découverte du dataset (~30 min)

Chargement, types, premier aperçu. Repérage de la cible et des variables
sensibles évidentes (`age`, `personal_status_sex`, `foreign_worker`).

In [12]:
df = pd.read_csv(RAW_PATH)

# Cible
display(df["credit_risk"].value_counts(normalize=True).rename("proportion").to_frame())

# Variables sensibles - age, personal_status_sex, foreign_worker
sensitive = ["age", "personal_status_sex", "foreign_worker"]
for col in sensitive: 
    display(df[col].value_counts().rename(col).to_frame())

,proportion
credit_risk,
good_credit,0.7
bad_credit,0.3


,age
age,
27,51
26,50
23,48
24,44
28,43
25,41
30,40
35,40
36,39


,personal_status_sex
personal_status_sex,
male single,548
female divorced/separated/married,310
male married/widowed,92
male divorced/separated,50


,foreign_worker
foreign_worker,
yes,963
no,37


**Observation** : `personal_status_sex` mélange genre et statut marital dans une seule variable composite. 
`foreign_worker` présente un fort déséquilibre (963 yes vs 37 no).

## 2. Audit qualité (~1 h)

Manquants, outliers, types, cohérence des modalités. **Au moins 4
visualisations** : distribution cible, distribution `age`, distribution
`credit_amount`, crosstab sensible (ex. `credit_risk` × `personal_status_sex`).

> *Quels problèmes de qualité ai-je identifiés ?* — court paragraphe à
> rédiger ici.

In [ ]:
# TODO — manquants par colonne
# df.isna().sum().sort_values(ascending=False)

In [ ]:
# TODO — visualisation 1 : distribution de la cible

In [ ]:
# TODO — visualisations 2/3/4 (au moins 4 au total)

## 3. Audit éthique léger (~1 h)

Déséquilibre cible + **disparate impact** sur au moins 2 variables sensibles.
Règle des 4/5 : `DI < 0.8` ou `> 1.25` → signal.

> *Quels biais structurels ai-je détectés ?* — court paragraphe.

⚠️ Pas de mitigation à ce stade — on documente, on alerte. La mitigation,
c'est M7.

In [ ]:
# TODO — Disparate impact sur foreign_worker (par exemple) :
# DI = P(good_credit | foreign_worker=non) / P(good_credit | foreign_worker=oui)

In [ ]:
# TODO — Disparate impact sur personal_status_sex
# Note : variable composite — fais un constat même si l'interprétation est limitée

## 4. Choix de prétraitement (~30 min)

Pour chaque feature, décide : numérique (impute + scale), ordinale
(impute + ordinal encoding **avec ordre**), catégorielle (impute + onehot),
ou exclue (justifie).

Remplis les listes `NUMERIC_FEATURES`, `ORDINAL_FEATURES`,
`CATEGORICAL_FEATURES` dans `src/preprocess.py`.

## 5. Industrialisation Pipeline (~1 h 15)

Une fois `src/preprocess.py` complété, fitte le pipeline et sauve-le.

### 🎓 Pourquoi un `Pipeline` ? (définition — la fuite de données)

Tu pourrais imputer/encoder/scaler « à la main » en Pandas. On enferme tout
dans un `Pipeline` parce qu'un préprocesseur **apprend des paramètres sur les
données** : `SimpleImputer` retient une médiane, `StandardScaler` une moyenne
et un écart-type, `OneHotEncoder` la liste des modalités.

**Fuite de données** (*data leakage*) = laisser de l'information du jeu de
**test** influencer la préparation du jeu d'**entraînement** — typiquement
calculer ces paramètres sur *tout* le dataset avant de le splitter. Symptôme :
des scores de validation **anormalement optimistes** qui s'effondrent en prod.

Le `Pipeline` est l'outil qui rend l'étanchéité automatique : un seul `fit` sur
le train propage la discipline à toutes les transformations. **Ici tu n'as pas
de split** (on produit juste un dataset propre) → `fit_transform` sur tout le
dataset est OK. **Réflexe à garder** : *dès qu'un modèle entre en jeu, le
préprocesseur ne se `fit` que sur le train.* → mise en œuvre en **M4-B1**.

In [ ]:
import sys
sys.path.append("../src")
from preprocess import build_preprocessor, load_dataset

X, y = load_dataset(RAW_PATH)
preprocessor = build_preprocessor()
X_transformed = preprocessor.fit_transform(X)
print(f"Shape originale : {X.shape}")
print(f"Shape transformée : {X_transformed.shape}")

In [ ]:
import joblib
joblib.dump(preprocessor, "../src/pipeline.joblib", compress=3)

## 6. Persistance Parquet + datasheet (~45 min)

Sauve le dataset propre en Parquet (pyarrow) et complète `datasheet.md`
(7 sections Gebru, schéma documenté dans *Composition*).

In [ ]:
# TODO — sauve un DataFrame *propre* (post-imputation si tu veux) en Parquet :
# df_clean.to_parquet(CLEAN_PATH, engine="pyarrow", compression="snappy")

In [ ]:
# Vérification relecture
df_back = pd.read_parquet(CLEAN_PATH)
print(f"Relu : {df_back.shape}, types préservés : {df_back.dtypes.value_counts().to_dict()}")

csv_size = RAW_PATH.stat().st_size / 1024
pq_size = CLEAN_PATH.stat().st_size / 1024
print(f"CSV : {csv_size:.1f} Ko · Parquet : {pq_size:.1f} Ko (compression : {csv_size/pq_size:.1f}x)")

## 7. Synthèse audit.md (~30 min)

Complète `audit.md` : verdict qualité (3-5 problèmes) + verdict éthique
(2-3 alertes) + recommandations. Lisible par le DPO Klaus Eichmann
(pas de jargon scikit-learn).